<a href="https://colab.research.google.com/github/kitchauhcmus/nlp_answer_passage/blob/main/nlp_answer_passage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Block 0: Cài đặt Thư viện và thiết lập môi trường

In [2]:
!pip install -q sentence-transformers rank_bm25 pandas numpy torch tqdm

from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/nlp_data_answer_passage


Mounted at /content/drive
/content/drive/MyDrive/nlp_data_answer_passage


### Block 1: Khai báo thư viện và Tiền xử lý Dữ liệu Cơ bản (Data Loading & Preprocessing)

In [6]:
import pandas as pd
import numpy as np
import torch
import re
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from rank_bm25 import BM25Okapi

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Đọc dữ liệu
corpus_df = pd.read_csv("corpus.csv").fillna("")
test_df = pd.read_csv("test.csv").fillna("")
sample_sub = pd.read_csv("sample_submission.csv")

# Tìm tên cột ID
corpus_id_col = "pid"
test_id_col = "qid"

# Tìm cột Nội dung
corpus_text_col = "text"
test_text_col = "question"

# 3. Trích xuất dữ liệu
passages = (corpus_df["title"] + " " + corpus_df[corpus_text_col]).tolist() # ghép tiêu đề vào nội dung
passage_ids = corpus_df[corpus_id_col].tolist()

questions = test_df[test_text_col].tolist()
question_ids = test_df[test_id_col].tolist()


### Block 2: Khởi tạo Hệ thống Truy xuất (BM25 & Dense)

In [7]:
# BM25 (Từ khóa)
tokenize = lambda s: re.findall(r"\w+", s.lower())
tokenized_corpus = [tokenize(doc) for doc in tqdm(passages, desc="Tokenize BM25")]
bm25 = BM25Okapi(tokenized_corpus) # Xây dựng lưới chỉ mục và tính trọng số BM25 (bản nâng cấp của TF-IDF)

# Dense (Ngữ nghĩa)
dense_model = SentenceTransformer("intfloat/multilingual-e5-base", device=DEVICE)
# Mã hóa toàn bộ kho 20.000 đoạn văn (thêm tiền tố passage: )
corpus_embeddings = dense_model.encode(
    ["passage: " + p for p in passages],
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True
)

# Mã hóa câu hỏi test (thêm tiền tố query: )
query_embeddings = dense_model.encode(
    ["query: " + q for q in questions],
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True
)


Tokenize BM25: 100%|██████████| 20000/20000 [00:01<00:00, 10363.88it/s]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

### Block 3: Kết hợp Lai và Xếp hạng lại (Hybrid Search & Re-ranking)

In [8]:
# Tải mô hình Reranker (bge-reranker-v2-m3)
# Khởi tạo Cross-Encoder với float16
ce = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=DEVICE,
                  automodel_args={"torch_dtype": torch.float16})

def rrf(lists, depth, k=60):
    """Thuật toán trộn hạng RRF cho 2 danh sách"""
    score = {}
    for lst in lists:
        for rank, i in enumerate(lst[:depth], 1):
            score[i] = score.get(i, 0.0) + 1.0 / (k + rank)
    return sorted(score, key=score.get, reverse=True)[:depth]

DEPTH = 50  # Lấy 50 đoạn văn đưa vào reranker
final_predictions = []

for i, q in enumerate(tqdm(questions, desc="Xử lý câu hỏi test")):
    # A. Tìm top 50 bằng BM25
    tok_q = tokenize(q)
    bm25_scores = bm25.get_scores(tok_q)
    bm25_top = np.argsort(bm25_scores)[::-1][:DEPTH]

    # B. Tìm top 50 bằng e5-base (so khớp vector)
    dense_scores = util.dot_score(query_embeddings[i], corpus_embeddings)[0]
    dense_top = torch.topk(dense_scores, DEPTH).indices.cpu().numpy()

    # C. Trộn hạng RRF lấy 50 ứng viên tổng hợp
    cands = rrf([dense_top, bm25_top], depth=DEPTH)

    # D. Xếp hạng lại bằng Cross-Encoder
    ce_inputs = [[q, passages[c]] for c in cands]
    ce_scores = ce.predict(ce_inputs, batch_size=DEPTH)

    # E. Chắt lọc Top 10 đoạn văn có điểm cao nhất
    top10_local_idx = np.argsort(ce_scores)[::-1][:10]
    top10_global_idx = [cands[idx] for idx in top10_local_idx]

    # Ánh xạ từ index nội bộ ra ID thực tế của đoạn văn
    top10_pids = [passage_ids[idx] for idx in top10_global_idx]

    # Ghép thành chuỗi cách nhau bằng dấu cách
    final_predictions.append(" ".join(map(str, top10_pids)))

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Xử lý câu hỏi test: 100%|██████████| 1508/1508 [28:30<00:00,  1.13s/it]


### BLOCK 4: Xuất file nộp bài

In [9]:
sub_df = pd.DataFrame({
    sample_sub.columns[0]: question_ids,
    sample_sub.columns[1]: final_predictions
})

sub_df.to_csv("submission_rerank_top50.csv", index=False)
